# Transformers for Computer Vision

> **Course: Computer Vision**  
> **Focus:** Self-attention · Vision Transformers (ViT) · Detection & hybrid architectures · CNN comparison

---

This notebook assumes you are already comfortable with **CNNs** (convolutions, residual blocks, detection heads). Here we connect that background to **Transformer-based vision** and run a pretrained **ViT** in PyTorch.



# Table of contents

0. [Setup](#setup)
1. [Vector Data Bases](#2)
2. [Transformer fundamentals (vision-oriented)](#2)
3. [Vision Transformer (ViT)](#3)
4. [SOTA and representative Transformer-style models](#4)
5. [Hands-on: load a pretrained ViT and run inference](#5)
6. [Loss Functions for Face Detection](#6)
7. [Optional: fine-tuning sketch](#7)
8. [Training MobileFaceNet using ArcFace as example](#8)
9. [References](#9)


<a id="0"></a>

# 0 - Setup

We use **PyTorch**, **torchvision**, and **timm** (large model zoo, including ViT variants). Install if needed, then select device.

### Imports, versions, and device
Imports PyTorch, `timm`, plotting, and HTTP utilities; selects `cuda` when available. Run the optional `pip` line in §1 first if anything is missing.



In [ ]:
!uv pip install psycopg2-binary torch torchvision timm matplotlib pillow requests facenet-pytorch insightface onnxruntime opencv-python-headless

In [ ]:
! git clone https://github.com/EzequielMatiasArevalo/tuia-computer-vision.git || ( cd tuia-computer-vision && git pull origin main)
! cp tuia-computer-vision/helpers/compose.yaml .

In [ ]:
!pip install udocker
!udocker --allow-root install
!udocker --allow-root pull  pgvector/pgvector:pg17

<a id="1"></a>

# 1 - Vector Data Bases

For this lab, we will use a PostgreSQL database to store vectors ( embeddings generated for our models ).


### 1.1 - What vector databases are

A **vector database** is a system designed to store, index, and query **embeddings**—numerical vectors that represent data such as text, images, audio, or code. Instead of exact matching (like SQL `=`), they enable **similarity search**: “find items most similar to this vector.”

Embeddings are typically produced by models (e.g., from OpenAI or Google), turning unstructured data into points in a high-dimensional space where **distance ≈ semantic similarity**.

---

*Docker cannot be installed directly in Google Colab* because Colab runs inside a restricted and ephemeral virtualized environment without full root-level access to the host system. Instead, we are going to use uDocker together with a PostgreSQL database.

*uDocker* is a lightweight tool that allows users to run Docker containers in environments where Docker itself cannot be installed or executed. It works entirely in user space, meaning it does not require administrator privileges or a Docker daemon. This makes it especially useful in platforms such as Google Colab, HPC clusters, and shared computing environments.

In [ ]:
!udocker --allow-root pull pgvector/pgvector:pg17

# Create persistent data directory
!rm -rf /var/lib/postgresql
!rm -rf postgres_data/
!mkdir -p /content/postgres_data

# Run container
!nohup udocker --allow-root run \
    -p=5432:5432 \
    -v=/content/postgres_data:/var/lib/postgresql/data \
    -e POSTGRES_USER=user \
    -e POSTGRES_PASSWORD=password \
    -e POSTGRES_DB=vector_db \
    pgvector/pgvector:pg17 \
&

In [ ]:
import time, psycopg2

# mirrors the healthcheck: interval 10s, retries 5 → we try 12× with 5s gaps
for attempt in range(12):
    time.sleep(5)
    try:
        conn = psycopg2.connect(
            host="localhost", port=5432,
            user="user", password="password", dbname="vector_db",
            connect_timeout=3,
        )
        # equivalent to the post_start psql -c "CREATE EXTENSION IF NOT EXISTS vector"
        with conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        conn.commit()
        conn.close()
        print("✅ PostgreSQL ready — pgvector extension installed!")
        break
    except Exception as e:
        print(f"⏳ attempt {attempt+1}/12 — {e}")
else:
    print("❌ could not connect after 60 s — check udocker logs above")

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import timm
import math
import requests
from io import BytesIO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | timm: {timm.__version__} | device: {device}")


## 1.2 - Core mechanics

* **Storage**: vectors (often 256–4096 dims) + metadata (IDs, tags, timestamps).
* **Indexing**: specialized structures to avoid brute-force scans:

  * HNSW (graph-based)
  * IVF / PQ (clustering + compression)
* **Query**:

  * k-NN (top-k nearest neighbors)
  * radius / range queries
  * hybrid (vector similarity + filters on metadata)
* **Distance metrics**: cosine similarity, dot product, Euclidean.

---

## 1.3 Typical pipeline

1. Ingest data → generate embeddings
2. Store vectors + metadata
3. Build/maintain index
4. Query with a new embedding → retrieve nearest items
5. (Optional) rerank with a cross-encoder

---

## 1.4 Where they’re used

* **RAG (Retrieval-Augmented Generation)** for LLMs
* Semantic search (documents, code)
* Recommendations (users/items)
* Deduplication / clustering
* Anomaly detection

Common systems: FAISS, Milvus, Pinecone, Weaviate.

---

## 1.5 Progress and evolution (why they matter now)

### 1. From research tools to production infrastructure

Early work (e.g., FAISS) focused on fast ANN search. Modern systems add:

* **Distributed scaling** (billions of vectors)
* **High availability** and replication
* **Managed services** (autoscaling, backups)

### 2. Hybrid retrieval is becoming default

Pure vector search is often not enough. Systems now support:

* **Vector + keyword (BM25)** fusion
* **Metadata filtering** (e.g., tenant, time, permissions)
* **Reranking pipelines** for precision

### 3. Better indexing and compression

Advances reduce latency/memory:

* HNSW optimizations (faster recall/latency trade-offs)
* Product quantization (PQ), scalar quantization
* Disk-backed indices (fit >RAM datasets)

### 4. Tight integration with LLM stacks

Vector DBs are now first-class in LLM workflows:

* Native connectors for frameworks
* Built-in **RAG primitives** (chunking, embedding, retrieval)
* Tooling around evaluation (recall@k, MRR)

### 5.) Multimodal embeddings

Not just text:

* Image, audio, video, and code embeddings
* Cross-modal search (e.g., text → image)

### 6. Real-time and streaming updates

* Low-latency inserts/updates
* Incremental indexing without full rebuilds
* Event-driven ingestion pipelines

### 7. Security and governance

* Row-level filtering / multi-tenancy
* Access control tied to metadata
* Auditability for enterprise use

### 8. Hardware and acceleration

* GPU acceleration for indexing/search
* SIMD/AVX optimizations on CPU
* Emerging support for specialized hardware

---

## Current limitations

* **Recall vs latency trade-off** (ANN is approximate)
* **Embedding quality dependency** (garbage in → garbage out)
* **Cost** at scale (memory, replication)
* **Evaluation complexity** (harder than exact-match systems)


In [ ]:
!pip install pgvector psycopg2-binary

In [ ]:
from pgvector.psycopg2 import register_vector
import psycopg2
from psycopg2.extras import execute_values
import numpy as np

DB_CONFIG = {
    "host":     "localhost",         # Cloud SQL: use /cloudsql/<instance>
    "port":     5432,
    "database": "vector_db",
    "user":     "user",
    "password": "password",
}

def get_connection():
    conn = psycopg2.connect(**DB_CONFIG)
    register_vector(conn)           # enables numpy <-> vector type conversion
    return conn


def setup_database():
    """
    Create tables and indexes.
    Run once on first setup.
    """
    conn = get_connection()
    cur  = conn.cursor()

    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

    # Main identities table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS identities (
            id          SERIAL PRIMARY KEY,
            person_id   TEXT UNIQUE NOT NULL,
            name        TEXT NOT NULL,
            created_at  TIMESTAMP DEFAULT NOW(),
            metadata    JSONB DEFAULT '{}'
        );
    """)

    # Face embeddings table — one row per image
    cur.execute("""
        CREATE TABLE IF NOT EXISTS face_embeddings (
            id          SERIAL PRIMARY KEY,
            identity_id INTEGER REFERENCES identities(id) ON DELETE CASCADE,
            embedding   vector(768),        -- ViT-B hidden dim
            image_path  TEXT,
            quality     FLOAT,              -- optional: AdaFace quality score
            created_at  TIMESTAMP DEFAULT NOW()
        );
    """)

    # ── Indexes ──
    # HNSW — best for approximate nearest neighbor at scale
    cur.execute("""
        CREATE INDEX IF NOT EXISTS face_emb_hnsw_idx
        ON face_embeddings
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64);
    """)

    conn.commit()
    cur.close()
    conn.close()
    print("Database setup complete ✓")

def save_embedding(
    identity_id: int,
    embedding: np.ndarray,
    image_path: str,
    quality: float = None,
):
    """Store a single face embedding."""
    conn = get_connection()
    cur  = conn.cursor()

    cur.execute("""
        INSERT INTO face_embeddings (identity_id, embedding, image_path, quality)
        VALUES (%s, %s, %s, %s);
    """, (identity_id, embedding, image_path, quality))

    conn.commit()
    cur.close()
    conn.close()

setup_database()

<a id="2"></a>

# 2- Transformer fundamentals (vision-oriented)

Transformers replace local convolutions with **global mixing** via **attention**. For images, the key ideas are the same as in NLP, but we often use **encoder-only** stacks (classification) or **encoder–decoder** (some generation / detection formulations).

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/attention.png" width=400>

## 2.1 Self-attention — intuition

For a sequence of tokens (vectors) $X \in \mathbb{R}^{n \times d}$, **scaled dot-product attention** compares every token to every other token:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/attention-4.png" width=600 height=400>

- **Query $Q$**: what am I looking for?

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/attention-1.png" width=600 height=400>

- **Key $K$**: what do I offer?

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/attention-3.png" width=600 height=400>

- **Value $V$**: what information do I pass if selected?

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/attention-5.png" width=600 height=400>

The softmax weights form an $n \times n$ **attention map**: each position distributes attention over all positions. Complexity is **$O(n^2)$** in sequence length — important for large images or many patches.

## 2.2 Multi-head attention (MHA)

One head might focus on texture, another on object parts. **Multi-head attention** runs several attention operations in parallel (with learned linear projections) and concatenates the results:

- **Benefit:** richer representation than a single pairwise pattern.
- **In code:** `nn.MultiheadAttention` (PyTorch) bundles the projections.


## 2.3 Positional information

Attention is **permutation-invariant** unless we inject **order**. Common options:

| Approach | Typical use |
|----------|-------------|
| **Sinusoidal** (fixed sin/cos) | Original Transformer, still used in variants |
| **Learned absolute embeddings** | ViT: one vector per patch index (+ class token) |
| **Relative biases** | Swin, some detection models |


## 2.4 Encoder / decoder (brief)

- **Encoder:** self-attention + FFN, builds a contextual representation of the input.
- **Decoder:** masked self-attention + cross-attention to encoder outputs (autoregressive models).
- **Vision:** ViT-like classifiers are **encoder-only**. Some detection setups use encoder–decoder or **DETR-style** object queries interacting with a CNN or Transformer backbone.


## 2.5 Minimal scaled dot-product attention

Educational single-head attention: compute \((QK^\top)/\sqrt{d}\), softmax, multiply by \(V\), then compare shapes with `nn.MultiheadAttention` (`batch_first=True`) on a token sequence.


In [ ]:
# Minimal scaled dot-product self-attention (single head) — educational
# Shapes: batch B, sequence n, model dim d


def scaled_dot_product_attention(q, k, v, dropout_p=0.0):
    """q, k, v: (B, n, d). Returns (B, n, d) and attention weights (B, n, n)."""
    d = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)  # (B, n, n)
    attn = torch.softmax(scores, dim=-1)
    if dropout_p > 0.0:
        attn = F.dropout(attn, p=dropout_p, training=True)
    out = torch.matmul(attn, v)
    return out, attn


B, n, d = 2, 5, 32
q = torch.randn(B, n, d)
k = torch.randn(B, n, d)
v = torch.randn(B, n, d)

out, attn = scaled_dot_product_attention(q, k, v)
print("output:", out.shape)  # (B, n, d)
print("attention weights row sums (should be ~1):", attn[0].sum(dim=-1))

# PyTorch built-in multi-head version (batch_first=True)
mha = nn.MultiheadAttention(embed_dim=128, num_heads=4, batch_first=True)
x = torch.randn(2, 16, 128)  # 16 tokens
attn_out, attn_weights = mha(x, x, x, average_attn_weights=False)
print("MHA out:", attn_out.shape, "| weights:", attn_weights.shape)


<a id="3"></a>

# 3- Vision Transformer (ViT)

**Idea (Dosovitskiy et al., 2020):** treat an image as a **sequence of patches**.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/VIT-0.png">


1. Split $H \times W$ image into $P \times P$ patches (e.g. $16 \times 16$).

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/VIT-1.png" width=700>

2. **Flatten + linearly embed** each patch → token of dimension $d$.
3. Prepend a **class token** (like BERT's `[CLS]`) or use **global average pooling** in some variants.
4. Add **positional embeddings** (learned 1D indices).

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/vit-2.png" width=800>

5. Stack **Transformer encoder** blocks (LayerNorm, MHA, MLP, residuals).
6. **Classification head** on the class token.



### 3.1 Training considerations & limitations

- **Data hunger:** ViT often needs **large-scale pretraining** (or strong augmentation + regularization) to match CNN sample efficiency on small datasets.
- **Inductive bias:** CNNs bake in **locality & translation equivariance**; ViT learns them from data — more flexible but sometimes slower to optimize on limited data.
- **Compute:** Self-attention is $O(n^2)$ in number of patches; high resolution → many patches → memory and latency grow quickly.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/vit-3.png" width=700>

## Summary

| Feature | CNN | ViT |
|---|---|---|
| **Processing unit** | Pixel neighborhoods — a small kernel (3×3 or 5×5) slides over every pixel. Understanding builds by stacking many local operations. | Patches — the image is cut into non-overlapping tiles (16×16 px by default) and each tile is treated as one token, exactly like a word in NLP. |
| **Inductive bias** | Strong (built-in). Locality: nearby pixels matter most. Translation equivariance: a cat in the corner activates the same filters as a cat in the center. The network doesn't need to learn these properties. | Weak (learned). No assumptions about spatial structure. The model must learn from data that neighboring patches are related — needs more data to compensate, but is more flexible across domains. |
| **Receptive field at layer 1** | Local (3×3 or 5×5 px). Global context only emerges after many layers: 3×3 at layer 1 → 5×5 effective field at layer 2 → full image only near the end. | Global (all patches at once). Self-attention connects every patch to every other in layer 1 already. A patch with an eye can immediately attend to the patch with the tail — no depth needed. |
| **Computational complexity** | O(n · k²) where n = pixels, k = kernel size. Cost scales nearly linearly with resolution — efficient even at high resolutions. | O(n²) where n = number of patches. Self-attention computes every pair of patches, so cost grows quadratically. A 224px image has 196 patches (fine); a 1024px image has 4,096 (expensive). Swin Transformer solves this with local windows. |
| **Data requirements** | Moderate. Strong inductive bias means the network converges well with ~1M images (e.g. ImageNet). Works even with tens of thousands when fine-tuning. | Large. The original paper needed JFT-300M to beat CNNs from scratch. DeiT later showed ImageNet-only training is possible via knowledge distillation. |
| **Scaling with data** | Saturates earlier. Performance gains plateau after a certain data volume — the inductive bias that helps with small data becomes a ceiling at large scale. | Keeps improving. Follows better scaling laws. With enough data and compute, ViT consistently outperforms CNNs at every model size. |
| **Cross-domain transfer** | Good for similar images. ImageNet-pretrained CNNs transfer well to natural image tasks but degrade more on distant domains (medical scans, satellite imagery). | More universal. ViT features (especially DINOv2) transfer across radically different domains without fine-tuning. The lack of inductive bias leads to more general representations. |
| **Interpretability** | Feature maps, CAM, GradCAM. Requires post-hoc tools — activations are not inherently human-readable. | Attention maps (more direct). Attention weights explicitly show which patches the model focused on. DINO showed that attention heads spontaneously learn to segment objects with no segmentation supervision. |
| **Training parallelism** | Limited by layer depth. Each conv layer must wait for the previous one — sequential by nature. | Fully parallel. All patches are processed simultaneously within each attention block, mapping perfectly onto GPU/TPU matrix-multiply units. |
| **State of the art (2024)** | ConvNeXt V2, EfficientNetV2. Modern CNNs borrowed ideas from ViT (LayerNorm, GELU, large kernels) and remain competitive on latency-constrained deployments. | ViT-G, DINOv2, SigLIP2, EVA-02. Dominate at scale. DINOv2 leads in self-supervised features; SigLIP2 powers Google Gemini's vision encoder. |

**When to prefer what (rule of thumb):**

- **Edge / real-time / small data:** often start with **CNN** or **CNN + light attention**.
- **Large pretraining budget / need global context:** **ViT or Swin** backbones; **DETR-like** if you want set-based end-to-end detection and can afford training complexity.

<a id="4"></a>

# 4- SOTA and representative Transformer-style models

## 4.1 DETR and variants

- **DETR** (Carion et al.): treats object detection as **set prediction** with a fixed number of **learned object queries**; the model reasons globally via Transformer layers. Training uses **bipartite matching** (Hungarian algorithm) between predictions and GT boxes.

- **Pros:** end-to-end, no hand-designed NMS anchor pipeline in the classical sense.

- **Variants (e.g. Deformable DETR, DINO):** improve convergence, efficiency, and accuracy with deformable attention, better training recipes, etc.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/detr.png">




## 4.2 DETR for panoptic segmentation

It combines two tasks that used to be separate:

```text
Semantic segmentation  → label every pixel with a class
                         ("this pixel is grass", "this pixel is sky")
                         does NOT distinguish between instances

Instance segmentation  → detect and mask individual objects
                         ("bird #1", "bird #2")
                         ignores background/stuff classes

Panoptic segmentation  → does BOTH at the same time
                         every pixel gets: class label + instance ID
```

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/detr2.png" >


## 4.3 Swin Transformer

- **Hierarchical** design: windows of self-attention + **shifted windows** to allow cross-window connections.
- **Complexity:** more favorable scaling than full global attention at full resolution.
- **Use:** strong backbone for classification, detection, segmentation.

## 4.4 Dino models

DINOv2 (Oquab et al., Meta AI, 2023) is a self-supervised ViT pre-training method that produces features so good they work as a frozen backbone for nearly any vision task — depth estimation, segmentation, classification, retrieval — without any fine-tuning. The key insight is that the teacher is the student's own past, so no human labels are ever needed.





## 4.5 Hybrid CNN–Transformer

- **Early hybrids:** CNN stem extracts local features; Transformer blocks model global context (e.g. **ConViT**, **CvT**, **CoAtNet**).

- **Practical motivation:** CNN front-end improves **sample efficiency** and **stability** while keeping long-range reasoning in later stages.




<a id="5"></a>

# 5- Hands-on: load a pretrained ViT and run inference

We use **timm** to load `vit_base_patch16_224` pretrained on ImageNet-1k, preprocess with the model's training recipe, and show **top-5** predictions.

## 5.1 ImageNet inference with a pretrained ViT
Fetches ImageNet-1k label strings, instantiates `vit_base_patch16_224` from `timm` with weights, builds the correct eval **preprocess** (`resolve_model_data_config`), loads an image from URL, and prints **top-5** softmax predictions.


In [ ]:
# ImageNet-1k class names (same ordering as torchvision / timm pretrained heads)
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL, timeout=30).text.strip().split("\n")
assert len(labels) == 1000

# Pretrained ViT-Base /16 at 224×224
model_name = "vit_base_patch16_224"
model = timm.create_model(model_name, pretrained=True)
model = model.to(device).eval()

# timm provides correct preprocessing for each model
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)
print("data_config:", {k: data_config[k] for k in ("input_size", "mean", "std") if k in data_config})


def load_image_from_url(url: str) -> Image.Image:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")


# Classic test image (replace with local path if offline)
img_url = "https://spectator.com/wp-content/uploads/2026/04/iStock-1460004637.jpg?w=724"
pil = load_image_from_url(img_url)
x = transform(pil).unsqueeze(0).to(device)

with torch.inference_mode():
    logits = model(x)
    probs = torch.softmax(logits, dim=-1)
    top5 = probs.topk(5, dim=-1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(pil)
ax.axis("off")
ax.set_title(f"{model_name} — top-1: {labels[top5.indices[0, 0]]}")
plt.show()

print("Top-5 predictions:")
for i in range(5):
    idx = int(top5.indices[0, i])
    p = float(top5.values[0, i])
    print(f"  {i+1}. {labels[idx]:40s}  {p:.4f}")


<a id="finetune"></a>
## 5.2 Optional: fine-tuning sketch (small dataset)

Fine-tuning usually means: **replace the classification head** with `num_classes`, optionally **freeze early layers** briefly, use a **smaller learning rate** for the backbone than the head, and train with standard augmentation.

Below we **download** the Kaggle celebrity-face dataset and **fine-tune** `vit_base_patch16_224` with a new classification head (CrossEntropy). Later cells add an optional **ArcFace** pipeline on the same `face_dataset_path`.


## 5.3 Plotting helper for many PIL images
Defines `plot_pil_imgs`: a small grid layout for previewing multiple images with optional per-image captions (used throughout the face demo).


In [ ]:

def plot_pil_imgs(pils : dict[str, Image.Image], descriptions: list[str] = [], color: str = "black", MAX_COLS: int = 2):

    n_cols = min(MAX_COLS, len(pils))
    n_rows = math.ceil(len(pils) / n_cols)
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(10, 10))

    if len(descriptions) > 0 and len(descriptions) != len(pils):
        raise ValueError(f"Descriptions must be the same length as pils, got {len(descriptions)} and {len(pils)}")

    for i, (k, pil) in enumerate(pils.items()):
        if i >= n_rows * n_cols:
            break
        if n_cols == 1:
            axs.imshow(pil)
        else:
            flat_axs = axs.flatten()
            flat_axs[i].imshow(pil)
            flat_axs[i].axis("off")
            title = f"{k} {descriptions[i]}" if len(descriptions) > 0 else k
            flat_axs[i].set_title(title, color=color)

    plt.tight_layout()
    plt.axis("off")
    plt.show()

<a id="6"></a>

# 6 - Models for Face recognition

## 6.1 Download and cache celebrity photos

We are going to download random images from the internet with different resolutions, poses, and viewing angles in order to evaluate our model, which was previously trained using the Celebraties dataset.

<b>This experiment will allow us to analyze how well our fine-tuned model generalizes to unseen data. We will also observe how differences in preprocessing and normalization techniques can negatively impact performance when the input images differ from those seen during training.</b>

In [ ]:
import os
output_dir = "media/dataset/celebrity-faces"
os.makedirs(output_dir + "/train", exist_ok=True)
os.makedirs(output_dir + "/val", exist_ok=True)
os.makedirs(output_dir + "/raw", exist_ok=True)
os.makedirs(output_dir + "/post-processed", exist_ok=True)

jl = {
    "jennifer_and_leo": "https://estaticos-cdn.prensaiberica.es/clip/efe46736-b143-4d34-bc45-13909135d4e1_16-9-discover-aspect-ratio_default_0.webp"
}

hj = {
    "wolverine": "https://fwmedia.fandomwire.com/wp-content/uploads/2025/09/01032952/hugh-jackman-wolverine.jpg?width=1600&height=900&fit=crop&format=auto&quality=70",
    "logan": "https://static0.polygonimages.com/wordpress/wp-content/uploads/2024/08/TDW-11260_R_be6c0b.jpg?q=49&fit=crop&w=1500&h=844&dpr=2",
    "hugh_semi": "https://static.wikia.nocookie.net/cine/images/b/b3/Hugh.jpg/revision/latest?cb=20200903224707",
    "hugh_x2": "https://archivo.prensa-latina.cu/wp-content/uploads/2023/03/thumb-hugh-jackman-wolverine-esteroides-1-Custom.jpg"
    }
scarlet = {
    "scarlet": "https://deadline.com/wp-content/uploads/2024/07/GettyImages-2161678100-e1739375514470.jpg?w=681&h=383&crop=1",
    "scarlet_hq": "https://phantom-telva.uecdn.es/4a71317644b245ad401fcd83a8e50934/resize/1280/f/webp/assets/multimedia/imagenes/2025/06/17/17501518079879.jpg",
    "scarlet_profile": "https://hips.hearstapps.com/hmg-prod/images/scarlett-johannson-attends-the-world-premiere-of-c2-a0jurassic-news-photo-1751578164.pjpeg?crop=0.573xw:0.860xh;0.274xw,0.0536xh"

}
pics = {**hj, **scarlet, **jl}
pils = {}
for k, url in pics.items():
    if not os.path.exists(output_dir + "/raw/" + k + ".jpg"):
        pil = load_image_from_url(url)
        pil.save(output_dir + "/raw/" + k + ".jpg")
        pils[k] = pil
    else:
        pils[k] = Image.open(output_dir + "/raw/" + k + ".jpg")

plot_pil_imgs(pils)



## 6.2 Face alignment and recognition (MTCNN + RetinaFace)

`align_face` uses **facenet-pytorch** MTCNN to get boxes and square crops.

`align_face_retinaface` uses **InsightFace** for detection scores, landmarks, optional embeddings, and cropped `PIL` images.

<a id='iface'></a>



## 6.3 MTCNN

MTCNN is a face detection and alignment pipeline built from three small CNNs arranged in a cascade. Each network refines the previous one's output — progressively filtering false positives and improving landmark localization.
It solves two tasks simultaneously in a single forward pass:

- 🔲 Face Detection — find bounding boxes around faces
- 📍 Face Alignment — locate 5 facial landmarks (eyes, nose, mouth corners)

This makes it the standard preprocessing step before any face recognition model (ArcFace, FaceNet, etc.).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

def align_face(img: Image.Image) -> list[Image.Image] | None:
    """
    Detect and align a face using MTCNN (facenet-pytorch).
    Returns a 112×112 PIL image or None if no face detected.

    For production use InsightFace's RetinaFace detector instead
    — it is more robust and supports 5-point landmark alignment.
    """
    try:
        from facenet_pytorch import MTCNN

        faces = []

        detector = MTCNN(image_size=112, margin=0, keep_all=True)
        #face = detector(img)
        boxes, probs = detector.detect(img)
        if boxes is None:
            return None
        # Convert back to PIL for use with transforms

        print(f"Detected {len(boxes)} faces")
        for i, (box, prob) in enumerate(zip(boxes, probs)):
            print(f"  Face {i+1}: box={box.astype(int)}, confidence={prob:.3f}")
            x1, y1, x2, y2 = box.astype(int)
            face = img.crop((x1, y1, x2, y2))
            #face_np = ((face.permute(1, 2, 0).numpy() + 1) * 127.5).astype(np.uint8)
            faces.append(face)
        return faces

    except ImportError:
        # Fallback: just resize without alignment (for demo)
        print("No MTCNN")
        return [img]
    except Exception as e:
        print(f"Error: {e}")
        return [img]

## 6.4 InsightFace

InsightFace is an open-source 2D & 3D deep face analysis library maintained by the team behind the ArcFace paper. It packages:

- Face detection (RetinaFace, SCRFD)
- Face recognition (ArcFace, CosFace)
- Face alignment (landmark-based affine transform)
- Face attribute analysis (age, gender, expression)
- 3D face reconstruction (optional)

*All of these are bundled into a single high-level object: FaceAnalysis.*

The Face Object Returned
Each detected face is a Face object — essentially a dictionary with dot-access:

```python
faces = app.get(img)
face  = faces[0]

face.bbox          # np.array [x1, y1, x2, y2]  — bounding box
face.det_score     # float                        — detection confidence (0–1)
face.kps           # np.array (5, 2)              — 5 landmark (x, y) coords
face.embedding     # np.array (512,)              — L2-normalized ArcFace embedding
face.sex           # str  'M' or 'F'              — gender (if genderage model loaded)
face.age           # int                          — estimated age (if loaded)
```


In [ ]:
import insightface
import cv2
from insightface.app import FaceAnalysis
from insightface.utils import face_align

app = FaceAnalysis(providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

def align_face_retinaface(
    img: Image.Image,
    norm_face: bool = True
) -> list[dict[str, any]] | None:

    cv2_img = cv2.cvtColor(np.array(img), cv2.COLOR_BGR2RGB)
    faces = app.get(cv2_img)

    results = []

    for face in faces:
        if norm_face:
          aligned = face_align.norm_crop(
              cv2_img,
              landmark=face.kps
          )
          aligned = cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB)

        results.append({
            "embedding": face.embedding,
            "bbox": face.bbox,
            "kps": face.kps,
            "det_score": face.det_score,
            "image": img.crop(face.bbox),
            "image_aligned": Image.fromarray(aligned) if norm_face else None
        })
    return results

## 6.5 Run face detection on all downloaded images
- Iterates `pils`
- runs MTCNN (`align_face`) and RetinaFace (`align_face_retinaface`)
-  builds `pil_faces` / `retina_faces`
- writes crops under `post-processed/`.


In [ ]:
import os

post_processed_folder= output_dir + "/post-processed-ds"
os.makedirs(post_processed_folder, exist_ok=True)
os.makedirs(post_processed_folder + "/val", exist_ok=True)
os.makedirs(post_processed_folder + "/test", exist_ok=True)
os.makedirs(output_dir + "/train", exist_ok=True)

pil_faces = {}
retina_faces = {}

for k, pil in pils.items():
    faces = align_face(pil)
    if faces is None:
        print(f"No face detected for {k}")
        pil_faces[k] = pils[k].resize((112, 112))
    else:
        for i, face in enumerate(faces):
            pil_faces[f"{k}_{i}"] = face

    r_faces = align_face_retinaface(pil)

    if r_faces is None:
        print(f"No face detected for {k}")
        retina_faces[k] = {
            "image": pils[k].resize((112, 112)),
            "bbox": None,
            "kps": None,
            "det_score": None
        }
        retina_faces[k]["image"].save(f"{output_dir}/test/{k}.jpg")
    else:
        for i, face in enumerate(r_faces):
            retina_faces[f"{k}_{i}"] = face
            face["image_aligned"].save(f"{post_processed_folder}/test/{k}_{i}.jpg")


## 6.6 Plot MTCNN crops
Displays face crops produced by `align_face` (MTCNN) for side-by-side comparison with RetinaFace.


In [ ]:
print("="*100)
print("Plotting pil faces with MTCNN")
plot_pil_imgs(pil_faces)


## 6.7 Visualize RetinaFace boxes and landmarks
Groups `retina_faces` by source image and overlays **red** bounding boxes and **blue** keypoints on the original RGB image for each detection.


In [ ]:
import matplotlib.patches as mpatches

print( "Plotting retina faces detections and kps")

group_by_image = {}
for k, v in retina_faces.items():
    bbox = v["bbox"]
    kps = v["kps"]
    original_image_key = "_".join(k.split("_")[:-1])
    if original_image_key not in group_by_image:
        group_by_image[original_image_key] = {
            "image": pils[original_image_key],
            "bboxes": [
                bbox
            ],
            "kps": [
                kps
            ]
        }
    else:
        group_by_image[original_image_key]["bboxes"].append(bbox)
        group_by_image[original_image_key]["kps"].append(kps)

for k, v in group_by_image.items():
    fig, ax = plt.subplots(figsize=(10, 10))

    for bbox, kps in zip(v["bboxes"], v["kps"]):
        rect = mpatches.Rectangle((bbox[0], bbox[1]), bbox[2] - bbox[0], bbox[3] - bbox[1], fill=False, color='red', linewidth=2)
        ax.add_patch(rect)
        ax.imshow(v["image"])
        for i, (x, y) in enumerate(kps):
            ax.scatter(x, y, color='blue', marker='o')
    plt.show()


## 6.7.1 Plot RetinaFace without align

In [ ]:
print("="*100)
print("Plotting retina faces with RetinaFace")
plot_pil_imgs({k: p["image"] for k, p in retina_faces.items() if p is not None})
print("="*100)

## 6.7.2 Plot RetinaFace-aligned crops
---

Shows every cropped face stored in `retina_faces` (InsightFace pipeline) but using align and normalization.

In [ ]:
print("="*100)
print("Plotting retina faces with RetinaFace")
plot_pil_imgs({k: p["image_aligned"] for k, p in retina_faces.items() if p is not None})
print("="*100)

## 6.8 Preparing Celebraties dataset

Kaggle data, splits, and dataloaders

- Downloads the celebrity face dataset with `kagglehub`
- wraps it in `ImageFolder`
- splits train/val (~90/10)
- applies ImageNet-style transforms (`MapSubset`)
- builds `DataLoader`s. Sets `num_classes` and `device` for training.


In [ ]:
# §7 — Fine-tune a pretrained ViT on the Celebrity Faces dataset (Kaggle)
# Requires: pip install kagglehub  (once)

from pathlib import Path

import kagglehub
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.datasets import ImageFolder
from torchvision import transforms

# --- 1) Download dataset (cached after first run) ---
dataset_root = Path("media/dataset/celebrity-faces/train")
dataset_root.mkdir(parents=True, exist_ok=True)
_downloaded = kagglehub.dataset_download(
    "vishesh1412/celebrity-face-image-dataset",
    output_dir=str(dataset_root),
    force_download=True
)
face_dataset_path = Path(_downloaded) / "Celebrity Faces Dataset"
assert face_dataset_path.is_dir(), f"Expected folder: {face_dataset_path}"
print("Path to dataset files:", face_dataset_path)
print("Classes:", [p.name for p in sorted(face_dataset_path.iterdir()) if p.is_dir()])


## 6.8.1 Create a post-processing training dataset
---

We created a new dataset using cropped faces from the celebrities dataset to train a model in order to achieve better performance.

In [ ]:
from IPython.lib.display import exists
from glob import glob
from datetime import datetime

os.makedirs(f"{post_processed_folder}/train", exist_ok=True )

for person in sorted(face_dataset_path.iterdir()):
    os.makedirs(f"{post_processed_folder}/train/{person.name}", exist_ok=True )
    raw_faces = glob(str(person / "*.jpg"))

    print(f"\n{person.name}", len(raw_faces))

    ETA = float("inf")
    for i,p in enumerate(raw_faces):

        name = str(person.name).lower().split(" ")[0]
        print(f"\rProcessing [", "="*i, " " * (len(raw_faces)-i), "]", f" ETA: {ETA:.2f}''", end="")
        start_time = datetime.now()
        img = Image.open(p)
        face = align_face_retinaface(img, norm_face=False)

        if not face:
          print(f"\nError : not face found for {p} . Moving to test folder !")
          img.save(f"{post_processed_folder}/test/{name}-{p.split('/')[-1]}")
          continue

        face = face[0]
        face["image"].save(f"{post_processed_folder}/train/{person.name}/{p.split('/')[-1]}")
        ETA = (datetime.now() - start_time).total_seconds() * (len(raw_faces) - i)

## 6.8.2 Defining normalization pipelines
These are the ImageNet dataset statistics, computed over the full ImageNet-1K training set.

```python
transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

```

All timm and torchvision ViT variants pretrained on ImageNet (including vit_base_patch16_224) expect input normalized with these exact values.

**Important:**
---

We are using cropped face images instead of pre-normalized or aligned faces.

This is because our custom dataset class already includes a transformation pipeline that performs normalization and data augmentation dynamically during the training phase.

Each time `__getitem__()` is called, the pipeline applies the corresponding preprocessing and augmentation steps on-the-fly.

This approach increases data variability during training and helps improve the model’s generalization capabilities.


In [ ]:

# --- 2) Transforms & train/val split (ImageNet normalization, 224×224 for ViT-B/16) ---
train_tf = transforms.Compose(
    [
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

val_tf = transforms.Compose(
    [
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

full_ds = ImageFolder(f"{post_processed_folder}/train/")
n_val = max(1, len(full_ds) // 10)
n_train = len(full_ds) - n_val
g = torch.Generator().manual_seed(42)
train_sub, val_sub = random_split(full_ds, [n_train, n_val], generator=g)


class MapSubset(Dataset):
    """Apply a transform to samples from a Subset (PIL images from ImageFolder)."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        img, y = self.subset[i]
        if self.transform is not None:
            img = self.transform(img)
        return img, y


train_ds = MapSubset(train_sub, train_tf)
val_ds = MapSubset(val_sub, val_tf)

num_classes = len(full_ds.classes)
batch_size = 32 if not torch.cuda.is_available() else 64
num_workers = 0  # avoid multiprocessing issues in some notebook environments

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

# --- 3) Model: replace head for num_classes; lower LR on backbone ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 6.9  Train the new head or load a checkpoint

If `models/model_ft.pth` exists, loads it and skips training.

Otherwise runs a short **AdamW** schedule (higher LR on `head`, lower on backbone), mixed-precision on CUDA, reports validation accuracy, and saves weights.

We fine-tune the model for *only 5 epochs* in this lab to illustrate the training process and to compare its embeddings with those from the same model without fine-tuning, in order to understand the differences.


In [ ]:
if os.path.exists("models/model_ft.pth"):
    model_ft = torch.load("models/model_ft.pth")
    finetune = False
else:
    model_ft = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=num_classes)
    finetune = True


model_ft = model_ft.to(device)

if finetune:
    print("Finetuning the model")

    optimizer = torch.optim.AdamW(
        [
            {"params": model_ft.head.parameters(), "lr": 3e-4},
            {
                "params": [p for n, p in model_ft.named_parameters() if not n.startswith("head.")],
                "lr": 3e-6,
            },
        ],
        weight_decay=0.05,
    )
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    FINETUNE_EPOCHS = 10

    for epoch in range(1, FINETUNE_EPOCHS + 1):

        start_time = datetime.now()

        model_ft.train()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                loss = criterion(model_ft(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * x.size(0)
        train_loss = running / n_train

        model_ft.eval()
        correct = 0
        total = 0
        with torch.inference_mode():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model_ft(x)
                correct += (logits.argmax(1) == y).sum().item()
                total += y.numel()
        val_acc = correct / total
        duration = (datetime.now() - start_time).total_seconds()
        ETA= duration * (FINETUNE_EPOCHS - epoch)
        print(f"Epoch {epoch}/{FINETUNE_EPOCHS}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f} time={duration:.2f}'' ETA={ETA:.2f}'' ")

    print("Fine-tuning (classification ViT) done. ")
    os.makedirs("models", exist_ok=True)
    torch.save(model_ft, "models/model_ft.pth")

## 6.10 Finetuned classifier: top-k on face crops

- Stacks RetinaFace crops into a batch
- runs the **fine-tuned classification** head
- prints top-3 class names per key.
- Overlays best class + probability as image titles via `plot_pil_imgs`.
- Requires `model_ft`, `val_tf`, and `full_ds`


In [ ]:


# --- Celebrity-class predictions (requires §7: `model_ft`, `val_tf`, `full_ds`, `device`) ---
class_names = full_ds.classes
batch = torch.stack([val_tf(retina_faces[k]["image"]) for k,pil in retina_faces.items() if pil is not None]).to(device)

model_ft.eval()
with torch.inference_mode():
    logits = model_ft(batch)

probs = torch.softmax(logits, dim=-1)
k_top = min(3, probs.size(-1))
top = probs.topk(k_top, dim=-1)

print(f"TOP: {len(top.indices)}")
print("Fine-tuned ViT — top predictions per URL:\n")
descriptions = []
for i, key in enumerate(retina_faces.keys()):
    print(f"{key}:")
    for r in range(k_top):
        idx = int(top.indices[i, r])
        print(f"  {r + 1}. {class_names[idx]:<22}  p={top.values[i, r].item():.4f}")

    best_match = class_names[top.indices[i, 0]]
    p = top.values[i, 0].item()
    descriptions.append(f"→ {best_match} {p:.2f}")

plot_pil_imgs({k: i["image"] for k,i in retina_faces.items()}, descriptions, MAX_COLS=2)




## 6.11 Generating and calculating embeddings for inference

The following cells wire **PostgreSQL + pgvector**: schema for identities and 768-D face embeddings, HNSW index for cosine search, and helpers to strip the ViT classifier head so the backbone acts as a **metric embedding** model for register / identify / verify flows.


## 6.12 ViT as a feature extractor

- Replaces `model_ft.head` with `Identity` so forward pass returns the **768-D** patch embedding.
- `get_embedding` applies `val_tf`
-  runs the model, and **L2-normalizes** for cosine similarity.


In [ ]:
# ── Remove the head in-place for inference
# Replaces model.head with Identity so forward() returns features
model_ft.head = nn.Identity()

@torch.no_grad()
def embed_v3(model, x):
    emb = model(x)                           # now returns (B, 768) directly
    return F.normalize(emb)

@torch.no_grad()
def get_embedding(
    model: nn.Module,
    img_or_path: Image.Image | str,
    device: torch.device,
) -> list[float]:
    if isinstance(img_or_path, str):
        img_pil = Image.open(img_or_path).convert("RGB")
    else:
        img_pil = img_or_path.convert("RGB") if img_or_path.mode != "RGB" else img_or_path
    #x = transform(img_pil).unsqueeze(0).to(device)  # (1, 3, 224, 224)
    x = val_tf(img_pil).unsqueeze(0).to(device)  # (1, 3, 224, 224)
    emb = embed_v3(model, x)  # (1, 768)
    return emb.squeeze(0).cpu().tolist()

## 6.13 Register identities and faces in Postgres
- `register_identity` upserts a row in `identities`
- `register_face` computes an embedding
- calls `save_embedding` to attach it to that identity.


In [ ]:
def register_identity(
    person_id: str,
    name: str,
    metadata: dict = None,
) -> int:
    """Create or update an identity record. Returns identity DB id."""
    conn = get_connection()
    cur  = conn.cursor()

    cur.execute("""
        INSERT INTO identities (person_id, name, metadata)
        VALUES (%s, %s, %s)
        ON CONFLICT (person_id)
        DO UPDATE SET name = EXCLUDED.name,
                      metadata = EXCLUDED.metadata
        RETURNING id;
    """, (person_id, name, psycopg2.extras.Json(metadata or {})))

    identity_id = cur.fetchone()[0]
    conn.commit()
    cur.close()
    conn.close()
    return identity_id

def register_face(
    model,
    device: torch.device,
    img_pil: Image.Image,
    person_id: str,
    person_name: str,
    metadata: dict = None,
    image_path: str | None = None,
):
    """Register one face image — creates identity if needed."""
    identity_id = register_identity(person_id, person_name, metadata)
    embedding   = get_embedding(model, img_pil, device)
    resolved = image_path or getattr(img_pil, "filename", None)
    if not resolved:
        raise ValueError(
            "register_face: pass image_path='...' when the PIL image has no .filename "
            "(e.g. loaded from URL or bytes)."
        )
    save_embedding(identity_id, embedding, resolved)
    print(f"Registered: {person_name} → identity_id={identity_id}")

## 6.14.1 Embedding Anlysys

### 6.14.1 Identification and verification API
- `identify_face` runs **1:N** search with pgvector cosine distance
- `verify_faces` compares two images **1:1**
- helpers fetch embeddings per `person_id`.
- Uses thresholds on cosine similarity.


In [ ]:
from dataclasses import dataclass

@dataclass
class FaceMatch:
    person_id:  str
    name:       str
    similarity: float
    image_path: str


def identify_face(
    model,
    device: torch.device,
    img_pil: Image.Image,
    top_k: int = 3,
    threshold: float = 0.5,
) -> dict:
    """
    1:N identification using cosine similarity via pgvector.

    The <=> operator = cosine distance (0=identical, 2=opposite)
    similarity = 1 - distance
    """
    query_emb = get_embedding(model, img_pil, device)

    conn = get_connection()
    cur  = conn.cursor()

    cur.execute("""
        SELECT
            i.person_id,
            i.name,
            1 - (fe.embedding <=> %s::vector) AS similarity,
            fe.image_path
        FROM face_embeddings fe
        JOIN identities i ON fe.identity_id = i.id
        ORDER BY fe.embedding <=> %s::vector   -- cosine distance ASC
        LIMIT %s;
    """, (query_emb, query_emb, top_k))

    rows = cur.fetchall()
    cur.close()
    conn.close()

    if not rows:
        return {"match": "unknown", "similarity": 0.0, "candidates": []}

    candidates = [
        FaceMatch(
            person_id=row[0],
            name=row[1],
            similarity=round(float(row[2]), 4),
            image_path=row[3],
        )
        for row in rows
    ]

    best = candidates[0]
    return {
        "match":      best.name if best.similarity >= threshold else "unknown",
        "similarity": best.similarity,
        "threshold":  threshold,
        "candidates": [
            {"name": c.name, "similarity": c.similarity}
            for c in candidates
        ],
    }


def verify_faces(
    model,
    device: torch.device,
    img_pil_1: Image.Image,
    img_pil_2: Image.Image,
    threshold: float = 0.5,
) -> dict:
    """1:1 verification — are these the same person?"""
    e1  = get_embedding(model, img_pil_1, device)
    e2  = get_embedding(model, img_pil_2, device)
    sim = float(np.dot(e1, e2))             # L2-normalized → dot = cosine

    return {
        "same_person": sim >= threshold,
        "similarity":  round(sim, 4),
        "threshold":   threshold,
    }


def get_identity_embeddings(person_id: str) -> list[np.ndarray]:
    """Retrieve all stored embeddings for a specific identity."""
    conn = get_connection()
    cur  = conn.cursor()

    cur.execute("""
        SELECT fe.embedding
        FROM face_embeddings fe
        JOIN identities i ON fe.identity_id = i.id
        WHERE i.person_id = %s;
    """, (person_id,))

    rows = cur.fetchall()
    cur.close()
    conn.close()
    return [np.array(row[0]) for row in rows]

### 6.14.2 Embedding Anlysys

We define a helper function to collect embeddings for our validation dataset.

In [ ]:
def collect_embeddings(loader, get_embeddings_fn, model, device, class_names):
    """Run get_embeddings over every batch in a DataLoader."""
    all_embs, all_labels = [], []

    for imgs, ys in loader:
        imgs = imgs.to(device)
        with torch.inference_mode():
            embs = get_embeddings_fn(model, imgs)          # (B, D) — L2-normalised
        all_embs.append(embs.cpu().numpy())
        all_labels.extend([class_names[y] for y in ys.tolist()])

    return np.vstack(all_embs), np.array(all_labels)


embeddings, labels = collect_embeddings(val_loader, embed_v3, model_ft, device, full_ds.classes)
print(f"Embeddings : {embeddings.shape}   |   Classes : {np.unique(labels)}")

### 6.14.3 Face Embeddings Visualization with PCA and t-SNE
This code performs dimensionality reduction and visualization of face embeddings using **PCA** and **t-SNE**.

Face embeddings are high-dimensional vector representations generated by a face recognition model.  

Each embedding encodes facial features in a numerical space where similar faces are located closer together.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder

# ── 1. Encode labels ───────────────────────────────────────────────────────
le       = LabelEncoder()
y_enc    = le.fit_transform(labels)
classes  = le.classes_
n_cls    = len(classes)
palette  = cm.get_cmap("tab10", n_cls)

# ── 2. PCA → 2-D ──────────────────────────────────────────────────────────
pca_2d      = PCA(n_components=2, random_state=42)
emb_pca     = pca_2d.fit_transform(embeddings)
var_exp     = pca_2d.explained_variance_ratio_ * 100

# Full PCA for scree
pca_full    = PCA(random_state=42).fit(embeddings)
cum_var     = np.cumsum(pca_full.explained_variance_ratio_) * 100
n_90        = int(np.searchsorted(cum_var, 90)) + 1     # components to reach 90 %

# ── 3. t-SNE → 2-D ────────────────────────────────────────────────────────
# Reduce to 50-D with PCA first (standard trick for large hidden_dim)
n_pre  = min(50, embeddings.shape[1], embeddings.shape[0] - 1)
emb_50 = PCA(n_components=n_pre, random_state=42).fit_transform(embeddings)

perp   = min(30, max(2, len(embeddings) // 4))           # safe for any N
tsne   = TSNE(n_components=2, perplexity=perp, n_iter=1200,
              init="pca", learning_rate="auto", random_state=42)
emb_tsne = tsne.fit_transform(emb_50)

In [ ]:
# ── 4. Dark-theme helpers ──────────────────────────────────────────────────

BG, GRID = "#111111", "#1e1e1e"

def _style(ax):
    ax.set_facecolor("#0e0e0e")
    ax.tick_params(colors="#888")
    ax.xaxis.label.set_color("#aaa")
    ax.yaxis.label.set_color("#aaa")
    ax.title.set_color("white")
    for sp in ax.spines.values():
        sp.set_edgecolor("#333")
    ax.grid(True, color=GRID, linewidth=0.6, zorder=0)


def _scatter(ax, coords, title, xlabel, ylabel,
             show_centroids=True, alpha=0.75, s=55):
    """Scatter with per-class colour, optional class centroids & legend."""
    # Points
    for c_idx in range(n_cls):
        mask = y_enc == c_idx
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            color=palette(c_idx), s=s, alpha=alpha,
            edgecolors="white", linewidths=0.4, zorder=3,
            label=classes[c_idx]
        )
    # Centroids
    if show_centroids:
        for c_idx in range(n_cls):
            mask  = y_enc == c_idx
            cx, cy = coords[mask, 0].mean(), coords[mask, 1].mean()
            ax.scatter(cx, cy, marker="*", s=260,
                       color=palette(c_idx), edgecolors="white",
                       linewidths=0.8, zorder=5)
            ax.annotate(
                classes[c_idx], (cx, cy),
                textcoords="offset points", xytext=(6, 4),
                fontsize=7, color="white",
                bbox=dict(boxstyle="round,pad=0.25", fc="black",
                          alpha=0.55, lw=0)
            )
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    _style(ax)
    ax.legend(
        fontsize=7, title="Class", title_fontsize=8,
        framealpha=0.25, facecolor="#1a1a1a", edgecolor="#444",
        markerscale=1.3
    )


### 6.14.4 Face Embeddings Visualization with PCA

PCA → 2D Projection
---

```python
pca_2d      = PCA(n_components=2, random_state=42)
emb_pca     = pca_2d.fit_transform(embeddings)
var_exp     = pca_2d.explained_variance_ratio_ * 100
```

<b>What is PCA?</b>
---

Principal Component Analysis (PCA) is a linear dimensionality reduction technique.

It projects high-dimensional embeddings into a lower-dimensional space while preserving as much variance (information) as possible.



<b>Why PCA for Face Embeddings?</b>
---

Face embeddings may have hundreds of dimensions:

```python
(1000 samples, 512 dimensions)
```

PCA reduces them to:

```python
(1000 samples, 2 dimensions)
```

allowing visualization in a 2D plot.

Each point represents a face in 2D space.

Faces from the same person should ideally cluster together.


<b>Percentage of variance explained by each PCA component.</b>
---

Example:

```python
[35.2, 18.7]
```

Meaning:
- First component explains 35.2% of total variance
- Second component explains 18.7%

Total explained variance:

```python
53.9%
```

<b>Purpose</b>
---

This computes PCA using all components to analyze how much information is preserved as dimensions increase.

### 6.14.4 Face Embeddings Visualization with TSNE


<b>PCA Preprocessing Before t-SNE</b>
---

```python
n_pre  = min(50, embeddings.shape[1], embeddings.shape[0] - 1)
emb_50 = PCA(n_components=n_pre, random_state=42).fit_transform(embeddings)
```

Before applying t-SNE, embeddings are first reduced to 50 dimensions using PCA.

This is a common optimization technique.


<b>Why Reduce to 50 Dimensions First?</b>

Benefits:
- removes noise
- speeds up t-SNE computation
- improves stability
- reduces memory usage

t-SNE performs poorly on extremely high-dimensional data directly.

<b> T-SNE → 2D Projection</b>
---

```python
perp   = min(30, max(2, len(embeddings) // 4))

tsne   = TSNE(
    n_components=2,
    perplexity=perp,
    n_iter=1200,
    init="pca",
    learning_rate="auto",
    random_state=42
)

emb_tsne = tsne.fit_transform(emb_50)
```


<b> What is t-SNE? </b>
---

t-SNE (t-Distributed Stochastic Neighbor Embedding) is a nonlinear dimensionality reduction algorithm specialized for visualization.

Unlike PCA:
- PCA preserves global variance
- t-SNE preserves local neighborhood structure

This makes t-SNE excellent for visualizing clusters of faces.


<b> Why Use t-SNE for Face Embeddings?</b>

Face embeddings often lie on complex nonlinear manifolds.

t-SNE can reveal:
- identity clusters
- outliers
- mislabeled samples
- overlapping identities

better than PCA.


<b> Key Parameters</b>
---

`perplexity`
---

Controls neighborhood size.

```python
perp = min(30, max(2, len(embeddings)//4))
```

Adaptive choice:
- avoids invalid values for small datasets
- balances local/global structure

Typical range:
- 5 → very local
- 30 → balanced
- 50+ → more global

---

`n_iter=1200`
---

Number of optimization iterations.

Higher values:
- improve convergence
- produce cleaner clusters
- increase computation time

---

`init="pca"`
--
Initializes t-SNE using PCA projection instead of random initialization.

Benefits:
- faster convergence
- more stable results

---

`learning_rate="auto"`
---

Automatically selects an appropriate learning rate.

---

Final Output
---

`emb_tsne`
---

2D t-SNE representation.


This output is commonly used to generate scatter plots where:
- each point is a face embedding
- colors represent identities
- clusters indicate similar faces

---



In [ ]:

# ── 5. Main figure: PCA + t-SNE side by side ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
fig.patch.set_facecolor(BG)
fig.suptitle("Embedding Space — Fine-tuned ViT  (L2-normalised)",
             fontsize=13, fontweight="bold", color="white", y=1.01)

_scatter(axes[0], emb_pca,
         f"PCA  (2-D)",
         f"PC 1  ({var_exp[0]:.1f} % var)",
         f"PC 2  ({var_exp[1]:.1f} % var)")

_scatter(axes[1], emb_tsne,
         f"t-SNE  (perplexity={perp})",
         "Dim 1", "Dim 2")

plt.tight_layout()
plt.savefig("embedding_pca_tsne.png", dpi=150,
            bbox_inches="tight", facecolor=BG)
plt.show()

### 6.14.5 PCA vs t-SNE

| Feature | PCA | t-SNE |
|---|---|---|
| Type | Linear | Nonlinear |
| Speed | Fast | Slow |
| Preserves | Global variance | Local neighborhoods |
| Good for | Compression | Visualization |
| Interpretable axes | Yes | No |
| Cluster visualization | Moderate | Excellent |

---

# Expected Visualization Behavior

## Good Face Embeddings

You should observe:
- tight clusters per identity
- large separation between identities
- minimal overlap

---

## Poor Face Embeddings

You may observe:
- mixed identities
- overlapping clusters
- scattered samples
- noisy distributions

---

# Typical Pipeline

```text
Face Images
      ↓
Face Recognition Model
      ↓
512-D Embeddings
      ↓
PCA (50-D)
      ↓
t-SNE (2-D)
      ↓
Visualization & Cluster Analysis
```



In [ ]:
# ── 6. Scree plot ──────────────────────────────────────────────────────────
max_comp = min(30, len(pca_full.explained_variance_ratio_))
comp_idx = np.arange(1, max_comp + 1)

fig2, ax2 = plt.subplots(figsize=(9, 4))
fig2.patch.set_facecolor(BG)

ax2.bar(comp_idx, pca_full.explained_variance_ratio_[:max_comp] * 100,
        color="#4fc3f7", alpha=0.85, label="Per-component", zorder=3)
ax2.plot(comp_idx, cum_var[:max_comp], "o--",
         color="#ff8a65", lw=1.8, ms=5, label="Cumulative", zorder=4)
ax2.axhline(90, ls=":", color="#aaa", lw=1.2, zorder=2)
ax2.axvline(n_90, ls=":", color="#aaa", lw=1.2, zorder=2)
ax2.text(n_90 + 0.3, 91, f"{n_90} components → 90 %",
         color="#aaa", fontsize=8)

ax2.set_xlabel("Principal Component")
ax2.set_ylabel("Explained Variance (%)")
ax2.set_title("PCA Scree — ViT Embedding Space",
              fontweight="bold", color="white")
ax2.legend(fontsize=9, framealpha=0.3, facecolor="#1a1a1a", edgecolor="#444")
_style(ax2)
plt.tight_layout()
plt.savefig("scree.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()


In [ ]:
# ── 7. Cosine similarity heatmap (bonus — works well on L2-normalised vecs)
from matplotlib.colors import LinearSegmentedColormap

# per-class mean embeddings
class_means = np.stack([
    embeddings[y_enc == c].mean(axis=0) for c in range(n_cls)
])
class_means /= np.linalg.norm(class_means, axis=1, keepdims=True)
sim_matrix   = class_means @ class_means.T          # cosine sim (already L2)

cmap_heat = LinearSegmentedColormap.from_list(
    "heat", ["#0d0d0d", "#1565c0", "#42a5f5", "#ffffff"])

fig3, ax3 = plt.subplots(figsize=(max(5, n_cls), max(4, n_cls - 1)))
fig3.patch.set_facecolor(BG)

im = ax3.imshow(sim_matrix, cmap=cmap_heat, vmin=0, vmax=1)
ax3.set_xticks(range(n_cls)); ax3.set_xticklabels(classes,
    rotation=40, ha="right", fontsize=8, color="#ccc")
ax3.set_yticks(range(n_cls)); ax3.set_yticklabels(classes,
    fontsize=8, color="#ccc")

for i in range(n_cls):
    for j in range(n_cls):
        ax3.text(j, i, f"{sim_matrix[i, j]:.2f}",
                 ha="center", va="center",
                 fontsize=7, color="white" if sim_matrix[i, j] < 0.7 else "black")

plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04).ax.tick_params(colors="#aaa")
ax3.set_title("Per-class Cosine Similarity (mean embeddings)",
              color="white", fontweight="bold")
ax3.set_facecolor("#0e0e0e")
plt.tight_layout()
plt.savefig("cosine_heatmap.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 6.15 Register embeddings in the database
Loops over post-processed crops for Scarlett and Hugh, calls `register_face` to upsert identities and store **L2-normalized** ViT embeddings linked to file paths.


In [ ]:
import glob

scarlet_orig_ds_imgs = glob.glob(f"{post_processed_folder}/train/Scarlett Johansson/*")[:3]
hugh_orig_ds_imgs = glob.glob(f"{post_processed_folder}/train/Hugh Jackman/*")[:3]

for path in scarlet_orig_ds_imgs:
    img_pil = Image.open(path)
    register_face(
        model=model_ft,
        device=device,
        img_pil=img_pil,
        person_id="scarlet",
        person_name="Scarlet",
        metadata={"department": "Avengers", "nickname": "Black Widow"},
        image_path=path,
    )


for path in hugh_orig_ds_imgs:
    img_pil = Image.open(path)
    register_face(
        model=model_ft,
        device=device,
        img_pil=img_pil,
        person_id="H_Jackman",
        person_name="Hugh Jackman",
        metadata={"department": "x-men", "nickname": "Wolverine"},
        image_path=path,
    )




## 6.16 Query: new photo → detect → identify
- Downloads (or loads) a new Scarlett image
- runs RetinaFace, and calls `identify_face` on each crop to pick the best match above a similarity threshold.


In [ ]:
if not os.path.exists(f"{output_dir}/raw/scarlet_new.jpg"):
    new_scarlet = load_image_from_url("https://media.vogue.es/photos/685a5ce9e2b2d2bb12456a2c/2:3/w_2240,c_limit/GettyImages-2216112225.jpg")
    new_scarlet.save(f"{output_dir}/raw/scarlet_new.jpg")
else:
    new_scarlet = Image.open(f"{output_dir}/raw/scarlet_new.jpg")

plot_pil_imgs( { "new image": new_scarlet }, MAX_COLS=2)
test_faces = align_face_retinaface(new_scarlet)
print( f"Faces found in the image: {len(test_faces)}" )

scarlet_metadata = {}
for face in test_faces:
    result = identify_face(model_ft, device, face["image_aligned"], threshold=0.4)

    if result["match"] == "Scarlet" and result["similarity"] > 0.5:
        print(f"✅ Same person: {result['match']} ({result['similarity']})")
        scarlet_face = face["image"]
        scarlet_metadata["img"] = scarlet_face
        scarlet_metadata["embedding"] = face["embedding"]
        scarlet_metadata["similarity"] = result["similarity"]
        scarlet_metadata["match"] = result["match"]
    else:
        print(f"❌ Skipping face: {result['match']} ({result['similarity']})")




## 6.17 Verification against saved Scarlett embeddings
- Compares the query Scarlett crop to every `post-processed/scarlet_*` image using `verify_faces` (cosine on normalized embeddings).
- Confirms gallery images match the same identity.


In [ ]:
# ── Verify ──


scarlet_post_processed = glob.glob(f"{post_processed_folder}/test/scarlett-*")
hugh_post_processed = glob.glob(f"{post_processed_folder}/test/hugh-*")
jennifer_post_processed = glob.glob(f"{post_processed_folder}/test/jennifer-*")
leo_post_processed = glob.glob(f"{post_processed_folder}/test/leo-*")

def compare_faces(model, device, scarlet_face, post_processed_faces, description, threshold=0.5):
    print("="*50)
    print(description)
    print("="*50)
    for path in post_processed_faces:
        img_pil = Image.open(path)
        image_name = os.path.basename(path)
        v = verify_faces(model, device, scarlet_face, img_pil, threshold)
        if v["same_person"]:
            fig, axs = plt.subplots(1, 2, figsize=(10, 10))
            axs[0].imshow(scarlet_face)
            axs[0].set_title(f"Scarlett")
            axs[1].imshow(img_pil)
            axs[1].set_title(f"{v['similarity']:.4f} {image_name}")
            plt.show()
            print(f"✅ Same person: ({v['similarity']} {image_name})")
        else:
            print(f"❌ Different person: ({v['similarity']})")

compare_faces(model_ft, device, scarlet_face, scarlet_post_processed, "Comparing Scarlett against Scarlett")
compare_faces(model_ft, device, scarlet_face, hugh_post_processed, "Comparing Scarlett against Hugh")
compare_faces(model_ft, device, scarlet_face, jennifer_post_processed, "Comparing Scarlet against Jennifer")


## 6.18 Baseline ViT WITHOUT finetuning (ImageNet only) for face retrieval
- Loads the same architecture pretrained on ImageNet **without** your celebrity fine-tuning, strips the head to features, and repeats identification/verification.
- Compares how much domain-specific fine-tuning changes similarity scores.


In [ ]:
model_without_finetune = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=num_classes)
model_without_finetune = model_without_finetune.to(device)
model_without_finetune.eval()

model_without_finetune.head = nn.Identity()

new_scarlet_embedding = get_embedding(model_without_finetune, scarlet_face, device)

def check_scarlett(model, device, new_scarlet_embedding, post_processed_faces, description, threshold=0.55):
    print("="*50)
    print(description)
    print("="*50)


    for path in post_processed_faces:
        img_pil = Image.open(path)
        scarlett_embedding = get_embedding(model_without_finetune, img_pil, device)
        similarity = float(np.dot(new_scarlet_embedding, scarlett_embedding))
        if similarity > threshold:
            print(f"✅ Same person: ({similarity})")
            plt.imshow(img_pil)
            plt.title(f"{similarity:.4f}")
            plt.show()
        else:
            print(f"❌ Different person: ({similarity})")

check_scarlett(model_without_finetune, device, new_scarlet_embedding, scarlet_post_processed, "Checking Scarlett")
check_scarlett(model_without_finetune, device, new_scarlet_embedding, hugh_post_processed, "Checking Hugh")
check_scarlett(model_without_finetune, device, new_scarlet_embedding, jennifer_post_processed, "Checking Jennifer")


## 6.19 Baseline InsightFace
- We are going to use <a href="#iface">Insight face</a> defined in previous section for embedding comparisson.

In [ ]:
hugh_imgs = glob.glob(f"{output_dir}/raw/hugh_*")
scarlet_imgs = glob.glob(f"{output_dir}/raw/scarlet_*")
jennifer_imgs = glob.glob(f"{output_dir}/raw/jennifer_*")

def l2_norm(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

def check_scarlett_insightface(new_scarlet_embedding, post_processed_faces, description, threshold=0.4):
    print("="*50)
    print(description)
    print("="*50)

    new_scarlet_embedding = l2_norm(new_scarlet_embedding)

    for path in post_processed_faces:
        img_pil = Image.open(path)
        face_embeddings = align_face_retinaface(img_pil)

        if not face_embeddings:
            print(f"❌ No face found in {path}")
            continue

        ncols = min(3, len(face_embeddings))
        nrows = (len(face_embeddings) + ncols - 1) // ncols
        figure, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
        axes = np.array(axes).reshape(nrows, ncols)

        for i, face_embedding in enumerate(face_embeddings):
            row = i // ncols
            col = i % ncols
            axes[row, col].imshow(face_embedding["image"])
            axes[row, col].axis("off")

            face_embedding["embedding"] = l2_norm(face_embedding["embedding"])
            similarity = float(np.dot(new_scarlet_embedding, face_embedding["embedding"]))

            if similarity > threshold:
                axes[row, col].set_title(f"Same person: ({similarity:.4f})", color="green")
            else:
                axes[row, col].set_title(f"Different person: ({similarity:.4f})", color="red")

        # Hide unused subplot slots
        for j in range(len(face_embeddings), nrows * ncols):
            row = j // ncols
            col = j % ncols
            axes[row, col].axis("off")

        plt.tight_layout()
        plt.show()

check_scarlett_insightface(scarlet_metadata["embedding"], scarlet_imgs, "Checking Scarlett")
check_scarlett_insightface(scarlet_metadata["embedding"], hugh_imgs, "Checking Hugh")
check_scarlett_insightface(scarlet_metadata["embedding"], jennifer_imgs, "Checking Jennifer")

<a id="7"></a>

#7 - Loss Functions for Face Detection

## 7.1 Softmax Loss

- *Discriminative Features*: Softmax loss may not yield features that are sufficiently discriminative for recognizing faces outside the training dataset. This is because softmax loss is optimized to minimize classification error on the training data, but it doesn’t explicitly encourage the model to learn features that generalize well to unseen faces.

- *Model Complexity and Memory*: As the number of identities in the training dataset increases, the size of the linear transformation matrix (typically the final layer of the neural network) also increases linearly. This leads to a larger model with higher computational and memory requirements. Managing such large models can be challenging, especially in real-time applications or resource-constrained environments.


## 7.2 Triplets Loss

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/triplets-loss.png">

 - *Local Comparisons*: Triplet loss focuses on comparing samples within a mini-batch, which means it only considers local relationships. This can sometimes limit its ability to capture the global structure of the data.

- *Increased Iteration Steps*: Since triplet loss requires considering all possible combinations of face triplets within each mini-batch, it leads to a significant increase in the number of iteration steps during training. This can result in longer training times, especially when dealing with large-scale datasets.

- *Triplet Selection*: One of the critical aspects of training with triplet loss is selecting appropriate face triplets, especially semi-hard samples. Identifying these triplets that provide meaningful gradients for model updates can be challenging, and inappropriate triplet selection may result in slower convergence or suboptimal performance.

- *Margin Tuning*: The choice of margin (the threshold that defines the minimum acceptable distance between anchor-positive and anchor-negative pairs) is crucial in triplet loss. Tuning this margin parameter can be non-trivial and may require careful experimentation to achieve optimal performance.

- *Memory Requirements*: Storing all possible combinations of face triplets during training can impose memory constraints, particularly for large datasets. Efficient triplet mining strategies and batch construction techniques are often employed to alleviate this issue.

## 7.3 ArcFace Loss Function


### 7.3.1 Motivation

Standard **Softmax Cross-Entropy** optimizes for correct classification but does **not** explicitly enforce geometric structure in the embedding space. As a result:

- Embeddings from different identities can **overlap** in feature space.
- The learned decision boundaries have **no guaranteed angular margin**.
- Verification performance (cosine similarity threshold) is suboptimal.

**ArcFace** solves this by adding an **additive angular margin** `m` directly in the angular (cosine) space, forcing the model to learn embeddings that are:

- **Intra-class compact** — same identity clusters tightly.
- **Inter-class separable** — different identities are pushed far apart.

---


### 7.3.2 Mathematical Formulation

### Standard Softmax Loss

$$L_{softmax} = -\log \frac{e^{W_{y_i}^T x_i + b_{y_i}}}{\sum_{j=1}^{N} e^{W_j^T x_i + b_j}}$$

### Normalized Softmax (CosFace baseline)

Fix biases to zero and L2-normalize weights and features:

$$W_j^T x_i = \|W_j\| \|x_i\| \cos\theta_j = \cos\theta_j \quad (\text{after normalization})$$

$$L_{norm} = -\log \frac{e^{s \cdot \cos\theta_{y_i}}}{\sum_{j=1}^{N} e^{s \cdot \cos\theta_j}}$$

where:
- $\theta_j$ = angle between embedding $x_i$ and class center $W_j$
- $s$ = feature scale (temperature parameter)

### ArcFace Loss

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/arcface-2-loss.png">

Add angular margin $m$ to the **target class angle only**:

$$\boxed{L_{ArcFace} = -\log \frac{e^{s \cdot \cos(\theta_{y_i} + m)}}{e^{s \cdot \cos(\theta_{y_i} + m)} + \sum_{j \neq y_i} e^{s \cdot \cos\theta_j}}}$$

where:
- $m \in [0, \pi)$ — additive angular margin (typically **0.5 radians ≈ 28.6°**)
- $s$ — feature scale (typically **64**)
- $\theta_{y_i}$ — angle between embedding and its **ground-truth** class center
- $N$ — total number of identities (classes)

---


### 7.3.3. Geometric Intuition

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_4_ViT_and_FD/arcface-loss.png">

- Without ArcFace: boundary sits at **θ = 90°**
- With ArcFace: the target logit uses **θ + m**, so the model must push **θ** smaller (i.e., embedding closer to center) to compensate
- This forces a **guaranteed angular margin of m** between the decision boundary and every class center

---



### 7.3.4 Step-by-Step Derivation

**Step 1 — L2-Normalize features and weights:**

$$\hat{x}_i = \frac{x_i}{\|x_i\|}, \quad \hat{W}_j = \frac{W_j}{\|W_j\|}$$

This maps all embeddings and class centers onto the **unit hypersphere** $S^{d-1}$.

**Step 2 — Compute cosine similarities:**

$$\cos\theta_j = \hat{W}_j^T \hat{x}_i \quad \in [-1, 1]$$

**Step 3 — Compute sin from cosine (Pythagorean identity):**

$$\sin\theta_{y_i} = \sqrt{1 - \cos^2\theta_{y_i}}$$

**Step 4 — Apply angle addition formula for target class:**

$$\cos(\theta_{y_i} + m) = \cos\theta_{y_i} \cos m - \sin\theta_{y_i} \sin m$$

This is computed in closed form — no need to call `arccos`.

**Step 5 — Numerical stability clamp:**

When $\theta_{y_i} + m > \pi$, $\cos(\theta_{y_i} + m)$ becomes larger than $\cos\theta_{y_i}$, which would *help* rather than *penalize*.  
Guard with:

$$\phi = \begin{cases} \cos(\theta_{y_i} + m) & \text{if } \cos\theta_{y_i} > \cos(\pi - m) \\ \cos\theta_{y_i} - m\sin(\pi - m) & \text{otherwise} \end{cases}$$

**Step 6 — Construct final logits and apply cross-entropy:**

$$\text{logit}_j = \begin{cases} s \cdot \phi & j = y_i \\ s \cdot \cos\theta_j & j \neq y_i \end{cases}$$

$$L = \text{CrossEntropy}(\text{logits}, y_i)$$




### 7.3.5 PyTorch Implementation


In [ ]:


import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
# importing optimizers

class ArcFaceLoss(nn.Module):
    """
    ArcFace: Additive Angular Margin Loss for Deep Face Recognition.

    Args:
        in_features  (int):   Embedding dimensionality (e.g., 512).
        num_classes  (int):   Number of identity classes.
        scale        (float): Feature scale s. Default: 64.0.
        margin       (float): Angular margin m in radians. Default: 0.5.
        easy_margin  (bool):  Simplified margin clipping. Default: False.
    """

    def __init__(
        self,
        in_features: int,
        num_classes: int,
        scale: float = 64.0,
        margin: float = 0.5,
        easy_margin: bool = False,
    ):
        super().__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.scale = scale
        self.margin = margin
        self.easy_margin = easy_margin

        # Class centers on the hypersphere (learnable)
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

        # Precompute margin trigonometry
        self.cos_m = math.cos(margin)          # cos(m)
        self.sin_m = math.sin(margin)          # sin(m)
        self.th    = math.cos(math.pi - margin)  # cos(π - m) — stability threshold
        self.mm    = math.sin(math.pi - margin) * margin  # sin(π - m) * m

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeddings: (B, in_features) — raw or pre-normalized face embeddings.
            labels:     (B,)             — integer identity indices [0, num_classes).

        Returns:
            Scalar ArcFace loss.
        """
        # ── Step 1: Normalize embeddings and class centers ──────────────────
        x = F.normalize(embeddings, p=2, dim=1)       # (B, D)
        W = F.normalize(self.weight,  p=2, dim=1)     # (C, D)

        # ── Step 2: Cosine similarities  cos θ_j ────────────────────────────
        cosine = F.linear(x, W)                       # (B, C)

        # ── Step 3: sin θ via Pythagorean identity ───────────────────────────
        sine = torch.sqrt(1.0 - cosine.pow(2).clamp(0.0, 1.0))

        # ── Step 4: cos(θ + m) = cos·cos_m − sin·sin_m ──────────────────────
        phi = cosine * self.cos_m - sine * self.sin_m  # (B, C)

        # ── Step 5: Stability clamp ──────────────────────────────────────────
        if self.easy_margin:
            # Only apply margin when cos θ > 0  (θ < 90°)
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            # Full guard: fallback to linear approximation when θ + m > π
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # ── Step 6: Build logits — phi for target class, cosine for others ───
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)

        logits = one_hot * phi + (1.0 - one_hot) * cosine
        logits *= self.scale                           # (B, C)

        # ── Step 7: Standard cross-entropy ───────────────────────────────────
        return F.cross_entropy(logits, labels)





---

## 7.4 Comparison with Related Losses

| Loss | Formula (target logit) | Margin Type | Notes |
|---|---|---|---|
| **Softmax** | $W^T x$ | None | No angular structure |
| **NormFace** | $s \cos\theta$ | None | Normalized, no margin |
| **SphereFace** | $s \cos(m\theta)$ | Multiplicative | Unstable; needs approx. |
| **CosFace** | $s(\cos\theta - m)$ | Additive cosine | Simpler, slightly weaker |
| **ArcFace** | $s \cos(\theta + m)$ | Additive angular | Best geometric interpretation |
| **MagFace** | $s \cos(\theta + m(r))$ | Adaptive angular | Assigns margins based on feature magnitude — links identity certainty to embedding norm. |
| **AdaFace** | $s \cos(\theta + m(q))$ | Quality-adaptive | margin based on image quality — better than ArcFace on low-quality/surveillance images. |



## 7.5 Decision Boundary Comparison

```text
SphereFace:   cos(m·θ) = cos θ_j  →  θ = π/(m+1)
CosFace:      cos θ - m = cos θ_j  →  no closed-form in angle space
ArcFace:      cos(θ + m) = cos θ_j →  θ = (π - m) / 2   (exact angular gap)
```

ArcFace is the only one with a **constant angular margin** in the hyperspherical space — making its geometry cleanest to reason about.

---


## 7.6 Hyperparameter Guide

| Param | Symbol | Typical Range | Effect |
|---|---|---|---|
| Scale | $s$ | 32 – 128 | Controls logit magnitude; too low → underfitting; too high → vanishing gradients |
| Margin | $m$ | 0.3 – 0.5 rad | Angular gap; larger = harder task; > 0.5 can cause instability |
| Embedding size | $d$ | 256 – 512 | 512 is standard; larger rarely helps |
| Num classes | $N$ | 1K – 100K+ | Scales with dataset; affects memory of weight matrix |


### Recommended Defaults

```python
# Standard benchmark setting (LFW, IJB-C)
scale  = 64.0
margin = 0.5

# Large-scale dataset (WebFace42M, MS1MV3)
scale  = 64.0
margin = 0.4    # slightly smaller to handle label noise

# Small dataset (< 10K identities)
scale  = 32.0
margin = 0.3    # easier task, smaller margin sufficient
```

---

## 7.7 Training Tips

### Optimizer

```python
# SGD generalizes better than Adam for ArcFace
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4
)

# LR schedule: cosine or step decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
# OR: drop at [20, 28, 32] epochs with factor 0.1 (common in InsightFace)
```




## 7.8 BatchNorm Before ArcFace

Always apply `BatchNorm1d` on the embedding **before** passing to ArcFace — it keeps the embedding magnitude consistent and stabilizes training:

```python
self.head = nn.Sequential(
    nn.Linear(2048, 512),
    nn.BatchNorm1d(512)    # ← critical; do NOT add ReLU here
)
```





---

## 7.9 Inference (No ArcFace Needed)

At inference, the ArcFace loss module is **discarded**. Only the backbone is used:

```python
model.eval()
with torch.no_grad():
    emb = backbone(face_image)              # (1, 512)
    emb = F.normalize(emb, p=2, dim=1)     # L2-normalize

# 1:1 Verification
cosine_sim = F.cosine_similarity(emb_a, emb_b).item()
same_person = cosine_sim > 0.45   # threshold tuned on val set

# 1:N Identification
scores = F.linear(emb, gallery_embeddings)  # (1, G)
best_idx = scores.argmax(dim=1)
```

<a id="8"></a>

# 8- Training MobileFaceNet using ArcFace as example

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MobileFaceNet(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        # simplified placeholder backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.PReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(64, embedding_dim)
        self.bn = nn.BatchNorm1d(embedding_dim)

    def forward(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = self.bn(x)
        x = F.normalize(x)  # critical for ArcFace
        return x

In [ ]:
from datetime import datetime

model = MobileFaceNet(embedding_dim=512)
criterion = ArcFaceLoss(512, num_classes=1000)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

TOTAL_EPOCHS=5
for epoch in range(TOTAL_EPOCHS):  # small lab example
    print(f"Epoch {epoch}")
    start_loss = None
    start_epoch = datetime.now()
    for i, (images, labels) in enumerate(train_loader):
        start_batch = datetime.now()
        embeddings = model(images)
        loss = criterion(embeddings, labels)

        if start_loss is None:
            start_loss = loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        end_batch = datetime.now()
        total_seconds = (end_batch - start_batch).total_seconds()
        batch_eta = total_seconds * (len(train_loader) - i)

        print(f"\rLoss: {loss.item()} | ETA: {batch_eta:.2f} seconds", end="")

    end_epoch = datetime.now()
    total_seconds = (end_epoch - start_epoch).total_seconds()
    print(f"\nStart loss: {start_loss} | End loss: {loss.item()} | Epoch: {total_seconds} | ETA: {total_seconds * (TOTAL_EPOCHS - epoch)} seconds")
    print("="*50)

# References

- Vaswani et al. | [*Attention Is All You Need*](https://arxiv.org/pdf/1706.03762)
- Dosovitskiy 2021 | [et al. *An Image is Worth 16x16 Words*](https://arxiv.org/pdf/2010.11929)
- Liu et al. | [*Swin Transformer*](https://arxiv.org/pdf/2103.14030)
- Carion et al. | [*DETR*](https://arxiv.org/pdf/2005.12872)
- Maxime Oquab 2024 | [DINOv2: Learning Robust Visual Features
without Supervision](https://arxiv.org/pdf/2304.07193)
- Jiankang Deng 2022 | [ArcFace: Additive Angular Margin Loss for Deep
Face Recognition](https://arxiv.org/pdf/1801.07698)
- Hao Wang 2018 | [CosFace: Large Margin Cosine Loss for Deep Face Recognition](https://arxiv.org/pdf/1801.09414)
- Jinghan You 2025 | [LVFace: Large Vision model for Face Recogniton](https://arxiv.org/html/2501.13420v1)